# 00 Setup: Create Virtual Environment & Verify Prerequisites

This notebook creates an isolated Python virtual environment for this demo and installs all required tools.

## 1. Create Virtual Environment

In [ ]:
import subprocess
import sys
from pathlib import Path

# Determine venv location (in demo root)
demo_root = Path("..").resolve()
venv_path = demo_root / ".venv"
is_windows = sys.platform == "win32"
venv_python = venv_path / ("Scripts" if is_windows else "bin") / ("python.exe" if is_windows else "python")

print(f"Demo root: {demo_root}")
print(f"Virtual environment path: {venv_path}\n")

# Create venv if it doesn't exist
if not venv_path.exists():
    print("Creating isolated virtual environment...")
    result = subprocess.run([sys.executable, "-m", "venv", str(venv_path)], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR creating venv: {result.stderr}")
        sys.exit(1)
    print("OK: Virtual environment created")
else:
    print("OK: Virtual environment already exists")

# Verify venv python executable exists
if not venv_python.exists():
    print("ERROR: Python executable not found")
    sys.exit(1)

print(f"OK: Virtual environment ready at {venv_path}")
print("\nInstalling ipykernel in .venv and registering notebook kernel...")

# Bootstrap ipykernel in the new venv before switching kernels
install_kernel = subprocess.run([str(venv_python), "-m", "pip", "install", "ipykernel"], capture_output=True, text=True)
if install_kernel.returncode != 0:
    print(f"ERROR installing ipykernel: {install_kernel.stderr}")
    sys.exit(1)

register_kernel = subprocess.run([
    str(venv_python),
    "-m",
    "ipykernel",
    "install",
    "--user",
    "--name",
    "evalgw02-venv",
    "--display-name",
    "Python (.venv) - evalgw02"
], capture_output=True, text=True)
if register_kernel.returncode != 0:
    print(f"ERROR registering kernel: {register_kernel.stderr}")
    sys.exit(1)

print("OK: Kernel registered as 'Python (.venv) - evalgw02'")

## 2. Switch Notebook Kernel to .venv (Required)

Now that `.venv` exists and kernel support is registered, switch the notebook kernel before continuing.

1. Open the kernel picker in the top-right of this notebook.
2. Choose `Python (.venv) - evalgw02` (or any interpreter from `.venv`).
3. If it is not visible yet: run `Developer: Reload Window`, then check the kernel picker again.
4. Return here and run the next cell to confirm.

In [ ]:
import sys
from pathlib import Path

python_path = Path(sys.executable).as_posix().lower()
print(f"Active Python: {sys.executable}")

if '/.venv/' in python_path or '\\.venv\\' in python_path:
    print('OK: Kernel is using .venv')
else:
    raise RuntimeError(
        "Kernel is not using .venv. Select 'Python (.venv) - evalgw02' in the kernel picker, then re-run this cell."
    )

## 3. Login to Azure (Defaults from repo root .env)

In [ ]:
import shutil
from pathlib import Path

print("Checking Azure CLI availability...")
az_path = shutil.which('az')
if not az_path:
    raise RuntimeError(
        "Azure CLI (az) was not found on PATH. Install Azure CLI and restart VS Code before continuing."
    )
print(f"OK: Azure CLI found at {az_path}")

# Read defaults from repo root .env (cross-demo settings)
DEFAULT_TENANT_ID = None
DEFAULT_SUBSCRIPTION_ID = None

demo_root = Path('..').resolve()
repo_root = demo_root.parents[1]
repo_env_file = repo_root / '.env'

if repo_env_file.exists():
    for raw_line in repo_env_file.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('\"').strip("'")
        if key == 'AZURE_TENANT_ID':
            DEFAULT_TENANT_ID = value or None
        elif key == 'AZURE_SUBSCRIPTION_ID':
            DEFAULT_SUBSCRIPTION_ID = value or None

print(f"Repo defaults file: {repo_env_file}")
print(f"Default tenant from repo .env: {DEFAULT_TENANT_ID or 'not set'}")
print(f"Default subscription from repo .env: {DEFAULT_SUBSCRIPTION_ID or 'not set'}")

In [ ]:
import subprocess
import json

print("Checking Azure CLI authentication status...")

# Read current account without triggering interactive login in the notebook
show_result = subprocess.run(
    'az account show --query "{SubscriptionId:id, Name:name, TenantId:tenantId}" -o json',
    shell=True,
    capture_output=True,
    text=True
)

if show_result.returncode != 0 or not show_result.stdout.strip():
    tenant_hint = f' --tenant \"{DEFAULT_TENANT_ID}\"' if DEFAULT_TENANT_ID else ''
    raise RuntimeError(
        "No active Azure CLI session found.\n"
        "Run this once in a terminal (outside the notebook), then re-run this cell:\n"
        f"az login --use-device-code{tenant_hint}"
    )

account = json.loads(show_result.stdout)
print(f"OK: Already logged in")
print(f"Active subscription: {account['Name']} ({account['SubscriptionId']})")
print(f"Active tenant: {account['TenantId']}")

# Apply default subscription from repo .env when present
if DEFAULT_SUBSCRIPTION_ID:
    if account['SubscriptionId'] != DEFAULT_SUBSCRIPTION_ID:
        print(f"\nSetting subscription from repo .env: {DEFAULT_SUBSCRIPTION_ID}")
        set_result = subprocess.run(
            f'az account set --subscription \"{DEFAULT_SUBSCRIPTION_ID}\"',
            shell=True,
            capture_output=True,
            text=True
        )
        if set_result.returncode != 0:
            tail = (set_result.stderr or '').splitlines()[-20:]
            raise RuntimeError('Failed to set subscription from repo .env.\n' + '\n'.join(tail))

        refresh_result = subprocess.run(
            'az account show --query "{SubscriptionId:id, Name:name, TenantId:tenantId}" -o json',
            shell=True,
            capture_output=True,
            text=True
        )
        if refresh_result.returncode == 0 and refresh_result.stdout.strip():
            account = json.loads(refresh_result.stdout)
            print(f"Active subscription: {account['Name']} ({account['SubscriptionId']})")
            print(f"Active tenant: {account['TenantId']}")

SUBSCRIPTION_ID = account['SubscriptionId']

## 4. Verify Subscription

In [ ]:
import subprocess
import sys
import json

# Apply default subscription from repo .env when present
if DEFAULT_SUBSCRIPTION_ID:
    print(f"Setting subscription from repo .env: {DEFAULT_SUBSCRIPTION_ID}")
    set_result = subprocess.run(
        f'az account set --subscription \"{DEFAULT_SUBSCRIPTION_ID}\"',
        shell=True,
        capture_output=True,
        text=True
    )
    if set_result.returncode != 0:
        tail = (set_result.stderr or '').splitlines()[-20:]
        raise RuntimeError('Failed to set subscription from repo .env.\n' + '\n'.join(tail))

# Show active subscription
result = subprocess.run(
    'az account show --query "{SubscriptionId:id, Name:name, TenantId:tenantId}" -o json',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0 or not result.stdout.strip():
    print('ERROR: Could not read active subscription. Run login step first.')
    sys.exit(1)

account = json.loads(result.stdout)
print(f"Active subscription: {account['Name']} ({account['SubscriptionId']})")
print(f"Active tenant: {account['TenantId']}")
SUBSCRIPTION_ID = account['SubscriptionId']

## 5. Install Azure Bicep

In [ ]:
import subprocess

# Install bicep
result = subprocess.run('az bicep install', shell=True, capture_output=True, text=True)
if result.returncode == 0 or 'already installed' in result.stderr:
    print('OK: Bicep is ready')
else:
    print(f"WARNING: {result.stderr}")

## 6. Install Python Packages

In [ ]:
import sys
import subprocess
from pathlib import Path

# Install from requirements.txt using the active kernel's Python
requirements_file = Path('../requirements.txt')
print(f'Requirements file: {requirements_file.resolve()}')

if not requirements_file.exists():
    raise RuntimeError(f'requirements.txt not found at {requirements_file.resolve()}')

print('Installing dependencies from requirements.txt...')
install_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)],
    capture_output=True,
    text=True
)

if install_result.returncode == 0:
    print('OK: All packages installed successfully')
else:
    tail = (install_result.stderr or '').splitlines()[-30:]
    raise RuntimeError('Package installation failed.\n' + '\n'.join(tail))

## Summary

✓ Setup completed with a single, required flow:
1. Create `.venv` and register `ipykernel`
2. Switch notebook kernel to `.venv`
3. Login to Azure and verify prerequisites
4. Install dependencies using the active `.venv` kernel

**Next Steps:**
1. Run `01_deploy_infra.ipynb`